In [293]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

In [294]:
base_dir = '/data/aman_singh/acuuracy_check'
channel = 'MT'
last_month = '2026-02-28'
run_month = '2026-03-31'

In [295]:
def list_all_files_in_directory(root):
    out = []

    for path, subdirs, files in os.walk(root):
        for name in files:
            out.append(os.path.join(path, name))

    return out

In [296]:
def discover_channel(file_path):
    #file_path = file_path.split('\\')[2]

    if 'ecom' in file_path:
        return 'ECOM'
    elif 'qcom' in file_path:
        return 'QCOM'
    elif 'mt' in file_path:
        return 'MT'
    elif 'gt' in file_path:
        return 'GT'
    else:
        return 'Channel not found'

    # return "MT"


In [297]:
discover_channel('/data/aman_singh/acuuracy_check/prophet_data_train_till_28_Feb_2026 (7)_mt.csv')

'MT'

In [298]:
def get_run_month(train_till_str):
    return datetime.strptime(train_till_str, 'train_till_%d_%b_%Y') + MonthEnd(1)

In [299]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [300]:
# query = f""" 
# SELECT 
#     channel_name,
#     asm_area_code, 
#     depot_code, 
#     parent_material1_code, 
#     month_date, 
#     sum(sec_actuals_vol_rum_month) Sec_Vol_Actuals_Rum_Month,
#     sum(sec_apo_plan_vol_rum_month) Sec_Vol_Apo_Plan_Rum_Month
# FROM (
#     Select 
#         Month_Date, 
#         Distributor_Code, 
#         material_code, 
#         sec_actuals_vol_rum_month, 
#         sec_apo_plan_vol_rum_month
#     from 
#         dwh_bpm_dist_brand_mth_sbp 
#     where 
#     month_date  between '2023-01-01' and '2026-02-28') A

# JOIN (
#     SELECT
#         channel_name, 
#         customer_code, 
#         asm_area_code, 
#         depot_code
#     FROM 
#         mst_customer 
#     WHERE 
#         company_code= 'MIL' 
#         and latest_record_ind=1 
#         and channel_name in ('MT', 'E-Commerce', 'Q-Commerce', 'GT')) C 
#     ON 
#         distributor_code = customer_code
# JOIN (
#     Select 
#         material_code, 
#         parent_material1_code 
#     from 
#         mst_material 
#     where 
#         company_code= 'MIL' 
#         and latest_record_ind=1) M 
#     ON 
#         A.material_code = M.material_code
#     group by 
#         channel_name, asm_area_code, depot_code, parent_material1_code, Month_date
#     ORDER BY 
#         channel_name, asm_area_code, depot_code, parent_material1_code, Month_date
# """
# # GT
# results = pd.read_sql(con=prod_conn, sql=query)
# sales_data = pd.DataFrame(results)
# sales_data.columns = sales_data.columns.str.lower()
# sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})

In [301]:
# query = f"""SELECT
#     CM.channel_name,
#     CM.asm_area_code,
#     CM.depot_code,
#     MM.parent_material_code,
#     LAST_DAY(MESR.month_date) AS month_date,
#     SUM(MESR.sec_apo_plan_vol_rum) AS Sec_Vol_Apo_Plan_Rum_Month,
#     SUM(MESR.sec_actuals_vol_rum) AS Sec_Vol_Actuals_Rum_Month
# FROM
#     dwh_bpm_dist_sku_daily MESR
# JOIN
# (
#     SELECT
#         material_code,
#         parent_material_code,
#         material_group_code,
#         uom_reporting,
#         vol_per_unit
#     FROM 
#         mst_material
#     WHERE
#         company_code='MIL' AND
#         latest_record_ind=1
        
# ) MM ON MESR.material_code = MM.material_code
# JOIN
# (
#     SELECT DISTINCT
#         channel_name, 
#         asm_area_code,
#         customer_code,
#         depot_code
#     FROM
#         mst_customer
#     WHERE
#         company_code='MIL' AND
#         latest_record_ind=1
#         AND channel_name = '{channel}'
# ) CM ON MESR.distributor_code = CM.customer_code
# WHERE
#     month_date BETWEEN '2023-01-01' and '{last_month}'
    
# GROUP BY 1, 2, 3, 4,5
# ORDER BY 1, 2, 3, 4,5
# """

# results = pd.read_sql(con=prod_conn, sql=query)
# sales_data = pd.DataFrame(results)
# sales_data.columns = sales_data.columns.str.lower()
# sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})
# sales_data

In [302]:
# extract data for all channels
query = f"""SELECT
    CM.channel_name,
    CM.asm_area_code,
    MM.material_group_code as brand_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(MESR.sec_apo_plan_vol_rum) AS Sec_Vol_Apo_Plan_Rum_Month,
    SUM(MESR.sec_actuals_vol_rum) AS Sec_Vol_Actuals_Rum_Month
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
        
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        channel_name, 
        asm_area_code,
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
        AND channel_name in ('E-Commerce', 'GT', 'MT', 'Q-Commerce') 
) CM ON MESR.distributor_code = CM.customer_code
WHERE
    month_date BETWEEN '2023-01-01' and '{last_month}'
    
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 3, 4
"""

results = pd.read_sql(con=prod_conn, sql=query)
sales_data = pd.DataFrame(results)
sales_data.columns = sales_data.columns.str.lower()
#sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})
sales_data

,channel_name,asm_area_code,brand_code,month_date,sec_vol_apo_plan_rum_month,sec_vol_actuals_rum_month
0,E-Commerce,BCE1,ADV-AHO-R,2023-01-31,2.2900,33.60
1,E-Commerce,BCE1,ADV-AHO-R,2023-02-28,4.6950,30.30
2,E-Commerce,BCE1,ADV-AHO-R,2023-03-31,2.2240,18.00
3,E-Commerce,BCE1,ADV-AHO-R,2023-04-30,22.9140,5.10
4,E-Commerce,BCE1,ADV-AHO-R,2023-05-31,31.8380,12.90
...,...,...,...,...,...,...
251919,Q-Commerce,QCW2,SW_SGPRF,2025-10-31,2.3902,0.00
251920,Q-Commerce,QCW2,SW_SGPRF,2025-11-30,7.0400,0.00
251921,Q-Commerce,QCW2,SW_SGPRF,2025-12-31,0.4700,20.88
251922,Q-Commerce,QCW2,SW_SGPRF,2026-01-31,0.7600,0.00


In [303]:
sales_data['channel_name'].unique()

array(['E-Commerce', 'GT', 'MT', 'Q-Commerce'], dtype=object)

In [304]:
# sales_data['key'] = (
#     sales_data['asm_area_code'].astype(str) + '_' +
#     sales_data['depot_code'].astype(str) + '_' +
#     sales_data['parent_material_code'].astype(str)
# )
# sales_data[sales_data['key'] == 'WMP_D464_718458']

In [305]:
# sales_data.groupby(['key'])['sec_vol_actuals_rum_month'].sum().reset_index().sort_values(by = ['sec_vol_actuals_rum_month'])

In [306]:
sales_data.isnull().sum()

channel_name                   0
asm_area_code                 47
brand_code                     0
month_date                     0
sec_vol_apo_plan_rum_month     0
sec_vol_actuals_rum_month      0
dtype: int64

In [307]:
sales_data = sales_data[sales_data['asm_area_code'].notna()]
sales_data.isnull().sum()

channel_name                  0
asm_area_code                 0
brand_code                    0
month_date                    0
sec_vol_apo_plan_rum_month    0
sec_vol_actuals_rum_month     0
dtype: int64

In [308]:
sales_data['month_date'].min()

datetime.date(2023, 1, 31)

In [309]:
realignment_df = pd.read_sql(
    'select * from trn_mil_asm_psku_realignment',
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

def realign_pskus(data, channel, columns=['sec_vol_actuals_rum_month', 'pri_actuals_vol_rum_month']):

    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "ALL")
    ]

    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
        old_psku, old_asm = grp
        new_psku = grp_data["psku new"].values[0]
        if old_asm != "ALL":
            condition = (data["parent_material_code"] == old_psku) & (
                data["asm_area_code"] == old_asm
            )
        else:
            condition = data["parent_material_code"] == old_psku

        data.loc[condition, "parent_material_code"] = new_psku


    data = data.groupby(
        ["channel_name", "asm_area_code", 'depot_code', "parent_material_code", "month_date"],
        as_index=False,
    )[columns].agg('sum')

    return data

In [310]:
sales_data['channel_name'].replace({'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'}, inplace=True)

In [311]:
sales_data['channel_name'].unique()

array(['ECOM', 'GT', 'MT', 'QCOM'], dtype=object)

In [312]:
sales_data['month_date'] = pd.to_datetime(sales_data['month_date'])

In [313]:
# realigned_df = pd.DataFrame()

# for channel in sales_data['channel_name'].unique():
#     tmp_df = sales_data[sales_data['channel_name'] == channel]
#     tmp_df = realign_pskus(tmp_df, channel=channel, columns=['sec_vol_actuals_rum_month'])
#     realigned_df = pd.concat([realigned_df, tmp_df])
#     del tmp_df

In [314]:
realigned_df = sales_data.copy()
realigned_df.duplicated(
    subset=['channel_name', 'asm_area_code', 'brand_code', 'month_date']
).sum()

0

In [315]:
realigned_df['key'] = realigned_df['asm_area_code'] + '_' + realigned_df['brand_code']
realigned_df['month_date'] = pd.to_datetime(realigned_df['month_date'])

realigned_df.duplicated(subset=['channel_name', 'key', 'month_date']).sum()

0

In [316]:
realigned_df = realigned_df.groupby(
    ['channel_name', 'key', 'asm_area_code', 'brand_code', 'month_date'],
    as_index=False
)['sec_vol_actuals_rum_month'].sum()

In [317]:
# realigned_df.to_csv('OT_data_debug.csv', index=False)

### Collate MIL

In [318]:
def collate_file(file_hint, extension='.csv'):
    collated_file = pd.DataFrame()

    run_path = f'{base_dir}'
    all_files = list_all_files_in_directory(run_path)

    for file_path in all_files:
        if file_hint in file_path:
            if extension == '.csv':
                print(file_path)
                read_file = pd.read_csv(file_path)
                read_file['channel'] = discover_channel(file_path)
                read_file['run'] = 'run'
                read_file['step'] = file_path.split('/')[3]
                read_file['file_path'] = file_path

                collated_file = pd.concat(
                    [collated_file, read_file]
                )
                del read_file

    return collated_file

In [319]:
channel

'MT'

In [320]:
trend_file_df = collate_file('trend_file_train_till')
prophet_file_df = collate_file('prophet_data_train_till')
# data_file_df = collate_file('\\data.csv')

/data/aman_singh/acuuracy_check/trend_file_train_till_28_Feb_2026 (7)_qcom.csv
/data/aman_singh/acuuracy_check/trend_file_train_till_28_Feb_2026 (7)_ecom.csv
/data/aman_singh/acuuracy_check/trend_file_train_till_28_Feb_2026 (7)_mt.csv
/data/aman_singh/acuuracy_check/trend_file_train_till_28_Feb_2026 (7)_gt.csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_28_Feb_2026 (7)_ecom.csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_28_Feb_2026 (7)_gt.csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_28_Feb_2026 (7)_mt.csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_28_Feb_2026 (7)_qcom.csv


In [321]:
trend_file_df.head()

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,brand_code,sec_vol_actuals_rum_month,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,qtr_ind_rate,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,drive,outlier
0,QCE1_ADV-AHO-R,2023-01-31,9.0,4.5,0.000000,9.84,0.000405,0.000203,0.000000,0.000443,QCOM,QCE1,ADV-AHO-R,9.0,0,0,0,0,0,0,450.07408,0.000405,NaN,NaN,9.0,0.000405,2026-02-28,1.816814,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN
1,QCE1_ADV-AHO-R,2023-02-28,9.0,4.5,0.000000,9.84,0.000405,0.000203,0.000000,0.000443,QCOM,QCE1,ADV-AHO-R,0.0,0,0,0,0,0,0,450.07408,0.000000,NaN,NaN,0.0,0.000000,2026-02-28,1.816814,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN
2,QCE1_ADV-AHO-R,2023-03-31,9.0,4.5,35.212431,35.52,0.000405,0.000203,0.001585,0.001599,QCOM,QCE1,ADV-AHO-R,18.0,0,0,0,0,0,0,450.07408,0.000810,NaN,NaN,18.0,0.000810,2026-02-28,1.816814,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN
3,QCE1_ADV-AHO-R,2023-04-30,9.0,4.5,0.000000,9.84,0.000405,0.000203,0.000000,0.000443,QCOM,QCE1,ADV-AHO-R,0.0,0,0,0,0,0,0,450.07408,0.000000,NaN,NaN,0.0,0.000000,2026-02-28,1.816814,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN
4,QCE1_ADV-AHO-R,2023-05-31,6.0,4.5,0.000000,5.40,0.000270,0.000203,0.000000,0.000243,QCOM,QCE1,ADV-AHO-R,0.0,0,0,0,0,0,0,450.07408,0.000000,NaN,NaN,0.0,0.000000,2026-02-28,1.816814,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN


In [322]:
trend_file_df['channel'].unique()

array(['QCOM', 'ECOM', 'MT', 'GT'], dtype=object)

In [323]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'brand_code',
       'sec_vol_actuals_rum_month', 'diwali', 'diwali_lead_1', 'diwali_lead_2',
       'ganesh_chaturthi', 'ganesh_chaturthi_lead_1',
       'ganesh_chaturthi_lead_2', 'qtr_ind_rate',
       'sec_vol_actuals_rum_month_value', 'pred_best_model',
       'pred_value_best_model', 'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov', 'run',
       'step', 'file_path', 'drive', 'outlier'],
      dtype='object')

In [324]:
trend_file_df['month_date'] = pd.to_datetime(trend_file_df['month_date'])
prophet_file_df['month_date'] = pd.to_datetime(prophet_file_df['month_date'])
# data_file_df['month_date'] = pd.to_datetime(data_file_df['month_date'])

trend_file_df['train_till'] = pd.to_datetime(trend_file_df['train_till'])
prophet_file_df['train_till'] = pd.to_datetime(prophet_file_df['train_till'])

trend_file_df['run_month'] = pd.to_datetime(trend_file_df['train_till']) + MonthEnd(1)
prophet_file_df['run_month'] = pd.to_datetime(prophet_file_df['train_till']) + MonthEnd(1)

assert (trend_file_df['run_month'] == pd.to_datetime(trend_file_df['train_till'] + MonthEnd(1))).all()
assert (prophet_file_df['run_month'] == pd.to_datetime(prophet_file_df['train_till'] + MonthEnd(1))).all()

In [325]:
trend_file_df[ (trend_file_df['channel'] == 'GT')
              & (trend_file_df['brand_code'] == 'NHR_AD_HO')].head(60)

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,brand_code,sec_vol_actuals_rum_month,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,qtr_ind_rate,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,drive,outlier,run_month
847,AURG_NHR_AD_HO,2023-01-31,17.636000,24.384833,14.520891,22.408767,0.363134,0.502096,0.298992,0.461408,GT,AURG,NHR_AD_HO,11.814,0,0,0,0,0,0,205905.0,0.243256,NaN,NaN,11.814,0.243256,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31
848,AURG_NHR_AD_HO,2023-02-28,17.636000,24.384833,20.751167,23.818400,0.363134,0.502096,0.427277,0.490433,GT,AURG,NHR_AD_HO,23.975,0,0,0,0,0,0,205905.0,0.493657,NaN,NaN,23.975,0.493657,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31
849,AURG_NHR_AD_HO,2023-03-31,17.636000,24.384833,15.879071,21.931900,0.363134,0.502096,0.326958,0.451589,GT,AURG,NHR_AD_HO,17.119,0,0,0,0,0,0,205905.0,0.352489,NaN,NaN,17.119,0.352489,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31
850,AURG_NHR_AD_HO,2023-04-30,17.636000,24.384833,27.020710,22.004033,0.363134,0.502096,0.556370,0.453074,GT,AURG,NHR_AD_HO,14.380,0,0,0,0,0,0,205905.0,0.296091,NaN,NaN,14.380,0.296091,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31
851,AURG_NHR_AD_HO,2023-05-31,18.491333,24.384833,36.776411,31.387400,0.380746,0.502096,0.757245,0.646282,GT,AURG,NHR_AD_HO,38.192,0,0,0,0,0,0,205905.0,0.786392,NaN,NaN,38.192,0.786392,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31
852,AURG_NHR_AD_HO,2023-06-30,23.230333,24.384833,35.831749,28.661400,0.478324,0.502096,0.737794,0.590153,GT,AURG,NHR_AD_HO,40.829,0,0,0,0,0,0,205905.0,0.840690,NaN,NaN,40.829,0.840690,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31
853,AURG_NHR_AD_HO,2023-07-31,31.133667,24.384833,17.515619,18.947467,0.641058,0.502096,0.360655,0.390138,GT,AURG,NHR_AD_HO,20.030,0,0,0,0,0,1,205905.0,0.412428,NaN,NaN,20.030,0.412428,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31
854,AURG_NHR_AD_HO,2023-08-31,33.017000,25.754167,19.262087,19.521467,0.679837,0.530291,0.396616,0.401957,GT,AURG,NHR_AD_HO,20.448,0,0,0,0,1,0,205905.0,0.421035,NaN,NaN,20.448,0.421035,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31
855,AURG_NHR_AD_HO,2023-09-30,27.102333,25.166333,23.576102,19.538333,0.558051,0.518187,0.485444,0.402304,GT,AURG,NHR_AD_HO,20.766,0,0,1,1,0,0,205905.0,0.427582,NaN,NaN,20.766,0.427582,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31
856,AURG_NHR_AD_HO,2023-10-31,20.414667,25.774167,18.249999,21.088400,0.420348,0.530703,0.375777,0.434221,GT,AURG,NHR_AD_HO,22.059,0,1,0,0,0,0,205905.0,0.454206,NaN,NaN,22.059,0.454206,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31


In [326]:
# data_file_df['run_month'] = pd.to_datetime(data_file_df['run_month'])

In [327]:
# data_file_df.columns

In [328]:
# missing_forecasts = pd.DataFrame()

# for rm_dt in trend_file_df['run_month'].unique():
#     for c in trend_file_df['channel'].unique():
#         tmp_df = data_file_df[
#             (data_file_df['run_month'] == rm_dt) &
#             (data_file_df['channel'] == c) &
#             (~data_file_df['key'].isin(trend_file_df[
#                 (trend_file_df['run_month'] == rm_dt) &
#                 (trend_file_df['channel'] == c)
#             ]['key'].unique()))
#         ]

#         missing_forecasts = pd.concat([missing_forecasts, tmp_df]).reset_index(drop=True)
    

In [329]:
# data_file_df.duplicated(
#     subset=['channel', 'run_month', 'month_date', 'asm_area_code',
#             'depot_code', 'parent_material_code']
# ).sum()

In [330]:
# missing_forecasts['tmp_key'] = missing_forecasts[['channel', 'key', 'run_month']].astype(str).agg('_'.join, axis=1)

In [331]:
# tmp_trend_df = trend_file_df.copy()
# tmp_trend_df['tmp_key'] = trend_file_df[['channel', 'key', 'run_month']].astype(str).agg('_'.join, axis=1)

# set(missing_forecasts['tmp_key'].unique()).intersection(tmp_trend_df['tmp_key'].unique())

In [332]:
# del tmp_trend_df, missing_forecasts['tmp_key']

In [333]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'brand_code',
       'sec_vol_actuals_rum_month', 'diwali', 'diwali_lead_1', 'diwali_lead_2',
       'ganesh_chaturthi', 'ganesh_chaturthi_lead_1',
       'ganesh_chaturthi_lead_2', 'qtr_ind_rate',
       'sec_vol_actuals_rum_month_value', 'pred_best_model',
       'pred_value_best_model', 'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov', 'run',
       'step', 'file_path', 'drive', 'outlier', 'run_month'],
      dtype='object')

In [334]:
trend_file_df[trend_file_df.duplicated(subset=['key', 'channel', 'month_date', 'run_month'])]

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,brand_code,sec_vol_actuals_rum_month,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,qtr_ind_rate,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,drive,outlier,run_month


In [335]:
trend_file_df['channel'].unique() #, prophet_file_df['channel'].unique()

array(['QCOM', 'ECOM', 'MT', 'GT'], dtype=object)

In [336]:
for channel in trend_file_df['channel'].unique():
    print(trend_file_df[trend_file_df['channel'] == channel]['asm_area_code'].unique())

['QCE1' 'QCE2' 'QCN1' 'QCN2' 'QCS1' 'QCS2' 'QCW1' 'QCW2']
['BCE1' 'BCE2' 'BCN1' 'BCN2' 'BCS1' 'BCS2' 'BCW1' 'BCW2' 'ECE1' 'ECE2'
 'ECN1' 'ECN2' 'ECS1' 'ECS2' 'ECW1' 'ECW2']
['BCE1' 'BCE2' 'BCN1' 'BCN2' 'BCS1' 'BCS2' 'BCW1' 'BCW2' 'MCE1' 'MCE2'
 'MCN1' 'MCN2' 'MCS1' 'MCS2' 'MCW1' 'MCW2']
['AURG' 'BIHE' 'BIHW' 'BLR' 'CMB' 'CNI' 'CTGH' 'CUP' 'DELM' 'EMP' 'EUP'
 'GUJN' 'GUJS' 'HAR' 'HYD1' 'HYD2' 'JHR' 'KARC' 'KARN' 'KOL' 'KRL' 'MUM1'
 'MUM2' 'NAG' 'NERE' 'NERW' 'NUP' 'ORS' 'PJB' 'PUNN' 'PUNS' 'RAJ1' 'RAJ2'
 'RWBN' 'RWBS' 'TPT' 'VIJ' 'WMP' 'WUP']


In [337]:
# trend_file_df.groupby(['key'])['sec_vol_actuals_rum_month'].sum().reset_index().sort_values(by = ['sec_vol_actuals_rum_month'])

In [338]:
trend_file_df.duplicated(subset=['key', 'channel', 'month_date', 'run_month']).sum(), prophet_file_df.duplicated(subset=['key', 'channel', 'month_date', 'run_month']).sum()

(0, 0)

In [339]:
mappings = {}

for run_month in trend_file_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 13):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-03-31 00:00:00'): {Timestamp('2026-03-31 00:00:00'): 'M',
  Timestamp('2026-04-30 00:00:00'): 'M+1',
  Timestamp('2026-05-31 00:00:00'): 'M+2',
  Timestamp('2026-06-30 00:00:00'): 'M+3',
  Timestamp('2026-07-31 00:00:00'): 'M+4',
  Timestamp('2026-08-31 00:00:00'): 'M+5',
  Timestamp('2026-09-30 00:00:00'): 'M+6',
  Timestamp('2026-10-31 00:00:00'): 'M+7',
  Timestamp('2026-11-30 00:00:00'): 'M+8',
  Timestamp('2026-12-31 00:00:00'): 'M+9',
  Timestamp('2027-01-31 00:00:00'): 'M+10',
  Timestamp('2027-02-28 00:00:00'): 'M+11',
  Timestamp('2027-03-31 00:00:00'): 'M+12'}}

In [340]:
trend_file_df['M month'] = trend_file_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

In [341]:
# missing_forecasts['M month'] = missing_forecasts.apply(
#     lambda x: mappings[x['run_month']].get(
#         x['month_date']
#     ), axis=1
# )

In [342]:
trend_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-03-31,2026-02-28


In [343]:
# prophet_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

In [344]:
trend_file_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7', 'M+8',
       'M+9', 'M+10', 'M+11', 'M+12'], dtype=object)

In [345]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,brand_code,sec_vol_actuals_rum_month,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,qtr_ind_rate,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,drive,outlier,run_month,M month
0,QCE1_ADV-AHO-R,2023-01-31,9.00,4.500000,0.000000,9.8400,0.000405,0.000203,0.000000,0.000443,QCOM,QCE1,ADV-AHO-R,9.0,0,0,0,0,0,0,450.074080,0.000405,NaN,NaN,9.0,0.000405,2026-02-28,1.816814,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,2026-03-31,None
1,QCE1_ADV-AHO-R,2023-02-28,9.00,4.500000,0.000000,9.8400,0.000405,0.000203,0.000000,0.000443,QCOM,QCE1,ADV-AHO-R,0.0,0,0,0,0,0,0,450.074080,0.000000,NaN,NaN,0.0,0.000000,2026-02-28,1.816814,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,2026-03-31,None
2,QCE1_ADV-AHO-R,2023-03-31,9.00,4.500000,35.212431,35.5200,0.000405,0.000203,0.001585,0.001599,QCOM,QCE1,ADV-AHO-R,18.0,0,0,0,0,0,0,450.074080,0.000810,NaN,NaN,18.0,0.000810,2026-02-28,1.816814,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,2026-03-31,None
3,QCE1_ADV-AHO-R,2023-04-30,9.00,4.500000,0.000000,9.8400,0.000405,0.000203,0.000000,0.000443,QCOM,QCE1,ADV-AHO-R,0.0,0,0,0,0,0,0,450.074080,0.000000,NaN,NaN,0.0,0.000000,2026-02-28,1.816814,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,2026-03-31,None
4,QCE1_ADV-AHO-R,2023-05-31,6.00,4.500000,0.000000,5.4000,0.000270,0.000203,0.000000,0.000243,QCOM,QCE1,ADV-AHO-R,0.0,0,0,0,0,0,0,450.074080,0.000000,NaN,NaN,0.0,0.000000,2026-02-28,1.816814,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,2026-03-31,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93569,WUP_SW_HR_WAX,2026-11-30,2.46,1.595833,0.000000,1.8655,0.000424,0.000275,0.000000,0.000322,GT,WUP,SW_HR_WAX,0.0,1,0,0,0,0,0,1725.124669,0.000000,0.000000,0.000000,0.0,0.000000,2026-02-28,0.937419,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,M+8
93570,WUP_SW_HR_WAX,2026-12-31,2.46,1.595833,1.447971,2.0880,0.000424,0.000275,0.000250,0.000360,GT,WUP,SW_HR_WAX,0.0,0,0,0,0,0,0,1725.124669,0.000000,1.447971,0.000250,0.0,0.000000,2026-02-28,0.937419,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,M+9
93571,WUP_SW_HR_WAX,2027-01-31,2.46,1.595833,0.000000,1.4805,0.000424,0.000275,0.000000,0.000255,GT,WUP,SW_HR_WAX,0.0,0,0,0,0,0,0,1725.124669,0.000000,0.000000,0.000000,0.0,0.000000,2026-02-28,0.937419,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,M+10
93572,WUP_SW_HR_WAX,2027-02-28,2.46,1.595833,0.371143,0.8785,0.000424,0.000275,0.000064,0.000152,GT,WUP,SW_HR_WAX,0.0,0,0,0,0,0,0,1725.124669,0.000000,0.371143,0.000064,0.0,0.000000,2026-02-28,0.937419,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,M+11


In [346]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'brand_code',
       'sec_vol_actuals_rum_month', 'diwali', 'diwali_lead_1', 'diwali_lead_2',
       'ganesh_chaturthi', 'ganesh_chaturthi_lead_1',
       'ganesh_chaturthi_lead_2', 'qtr_ind_rate',
       'sec_vol_actuals_rum_month_value', 'pred_best_model',
       'pred_value_best_model', 'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov', 'run',
       'step', 'file_path', 'drive', 'outlier', 'run_month', 'M month'],
      dtype='object')

In [347]:
prophet_file_df[
    ['month_date', 'key', 'channel', 'run_month']
].duplicated().sum()

0

In [348]:
prophet_file_df.columns

Index(['ds', 'trend', 'yhat_lower', 'yhat_upper', 'trend_lower', 'trend_upper',
       'yhat_60_%ile', 'yhat_70_%ile', 'yhat_75_%ile', 'trend_60_%ile',
       'trend_70_%ile', 'trend_75_%ile', 'additive_terms',
       'additive_terms_lower', 'additive_terms_upper', 'diwali',
       'diwali_lower', 'diwali_upper', 'diwali_lead_1', 'diwali_lead_1_lower',
       'diwali_lead_1_upper', 'diwali_lead_2', 'diwali_lead_2_lower',
       'diwali_lead_2_upper', 'extra_regressors_additive',
       'extra_regressors_additive_lower', 'extra_regressors_additive_upper',
       'ganesh_chaturthi', 'ganesh_chaturthi_lower', 'ganesh_chaturthi_upper',
       'ganesh_chaturthi_lead_1', 'ganesh_chaturthi_lead_1_lower',
       'ganesh_chaturthi_lead_1_upper', 'ganesh_chaturthi_lead_2',
       'ganesh_chaturthi_lead_2_lower', 'ganesh_chaturthi_lead_2_upper',
       'yearly', 'yearly_lower', 'yearly_upper', 'multiplicative_terms',
       'multiplicative_terms_lower', 'multiplicative_terms_upper', 'yhat',
     

In [349]:
# Merge 70th percentile Prophet predictions
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    prophet_file_df[['month_date', 'channel', 'key', 'run_month', 'yhat_60_%ile', 'yhat_70_%ile']].rename(
        columns={
            'yhat_70_%ile': 'pred_prophet_70%ile',
            'yhat_60_%ile': 'pred_prophet_60%ile'
        }
    ),
    on=['month_date', 'channel', 'key', 'run_month'],
    how='left'
)
assert len(trend_file_df) == len_before_merge
del len_before_merge

In [350]:
# trend_file_df['Skipped'] = 0

In [351]:
# missing_forecasts['Skipped'] = 1

In [352]:
# trend_file_df = pd.concat([trend_file_df, missing_forecasts]).reset_index(drop=True)

In [353]:
trend_file_df.duplicated(subset=['key', 'channel', 'month_date', 'run_month']).sum()

0

In [354]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")

In [355]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [356]:
qtr_ind_df = read_qtr_ind_rate_table()
qtr_ind_df.head()


Credentials retrieved successfully for prod db.


,month_date,brand_code,qtr_ind_rate
0,2026-03-31,PADV_WIPS,168.752000
1,2026-03-31,PABABY_SP,415.245000
2,2026-03-31,P_GOHR_SR,7470.000000
3,2026-03-31,PA_JAS_GD,298.847906
4,2026-03-31,PA_NOR_HO,188992.000000


In [357]:
trend_file_df['portfolio'].isna().sum()

0

In [358]:
assert qtr_ind_df.duplicated(subset=['brand_code']).sum() == 0
qtr_ind_df.drop('month_date', axis=1, inplace=True)

In [359]:
trend_file_df.drop('qtr_ind_rate', axis=1, inplace=True)

In [360]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    qtr_ind_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [361]:
assert trend_file_df.duplicated(
    subset=['run_month', 'month_date', 'channel', 'key']
).sum() == 0

In [362]:
#trend_file_df.drop('sec_vol_actuals_rum_month', axis=1, inplace=True)

In [363]:
realigned_df.dtypes

channel_name                         object
key                                  object
asm_area_code                        object
brand_code                           object
month_date                   datetime64[ns]
sec_vol_actuals_rum_month           float64
dtype: object

In [364]:
# assert realigned_df.duplicated(subset=['channel_name', 'key', 'month_date']).sum() == 0

In [365]:
# len_before_merge = len(trend_file_df)
# trend_file_df = trend_file_df.merge(
#     realigned_df[['channel_name', 'key', 'month_date', 'sec_vol_actuals_rum_month']].rename(
#         columns={
#             'channel_name': 'channel'
#         }
#     ),
#     on=['month_date', 'channel', 'key'],
#     how='left'
# )
# assert len(trend_file_df) == len_before_merge

In [366]:
trend_file_df.select_dtypes('number').isna().sum()

pred_p3m                                        0
pred_p6m                                        0
pred_prophet                                    0
pred_rf                                         0
pred_value_p3m                                  0
pred_value_p6m                                  0
pred_value_prophet                              0
pred_value_rf                                   0
sec_vol_actuals_rum_month                       0
diwali                                          0
diwali_lead_1                                   0
diwali_lead_2                                   0
ganesh_chaturthi                                0
ganesh_chaturthi_lead_1                         0
ganesh_chaturthi_lead_2                         0
sec_vol_actuals_rum_month_value                 0
pred_best_model                            129618
pred_value_best_model                      129618
sec_vol_actuals_rum_month_treated               0
sec_vol_actuals_rum_month_value_treated         0


In [367]:
trend_file_df = trend_file_df.fillna(0)

In [368]:
trend_file_df.select_dtypes('number').min().round()

pred_p3m                                        0.0
pred_p6m                                        0.0
pred_prophet                                    0.0
pred_rf                                         0.0
pred_value_p3m                                  0.0
pred_value_p6m                                  0.0
pred_value_prophet                              0.0
pred_value_rf                                   0.0
sec_vol_actuals_rum_month                       0.0
diwali                                          0.0
diwali_lead_1                                   0.0
diwali_lead_2                                   0.0
ganesh_chaturthi                                0.0
ganesh_chaturthi_lead_1                         0.0
ganesh_chaturthi_lead_2                         0.0
sec_vol_actuals_rum_month_value                 0.0
pred_best_model                                 0.0
pred_value_best_model                           0.0
sec_vol_actuals_rum_month_treated               0.0
sec_vol_actu

In [369]:
# 'pred_prophet_70%ile',
for col in [ 'pred_best_model', 'pred_value_best_model',  'sec_vol_actuals_rum_month', 'pred_prophet_70%ile', 'pred_prophet_60%ile']:
    trend_file_df[col] = trend_file_df[col].clip(lower=0)

In [370]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'channel', 'key', 'month_date']
)

In [371]:
trend_file_df['P3M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['P6M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

trend_file_df['LY P3M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['LY P6M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [372]:
trend_file_df[ (trend_file_df['channel'] == 'GT')
              & (trend_file_df['brand_code'] == 'NHR_AD_HO')].head(60)#['P3M'].sum()
#(trend_file_df['month_date'] == '2026-04-30') &
# realigned_df[ (realigned_df['channel_name'] == 'GT')
#               & (realigned_df['brand_code'] == 'NHR_AD_HO')].head(60)

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,brand_code,sec_vol_actuals_rum_month,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,drive,outlier,run_month,M month,pred_prophet_60%ile,pred_prophet_70%ile,portfolio,qtr_ind_rate,P3M,P6M,LY P3M,LY P6M
85901,AURG_NHR_AD_HO,2023-01-31,17.636000,24.384833,14.520891,22.408767,0.363134,0.502096,0.298992,0.461408,GT,AURG,NHR_AD_HO,11.814,0,0,0,0,0,0,0.243256,0.000000,0.000000,11.814,0.243256,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,16.025453,17.635468,Other VAHO,205905.0,NaN,NaN,NaN,NaN
85902,AURG_NHR_AD_HO,2023-02-28,17.636000,24.384833,20.751167,23.818400,0.363134,0.502096,0.427277,0.490433,GT,AURG,NHR_AD_HO,23.975,0,0,0,0,0,0,0.493657,0.000000,0.000000,23.975,0.493657,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,22.112487,23.463560,Other VAHO,205905.0,NaN,NaN,NaN,NaN
85903,AURG_NHR_AD_HO,2023-03-31,17.636000,24.384833,15.879071,21.931900,0.363134,0.502096,0.326958,0.451589,GT,AURG,NHR_AD_HO,17.119,0,0,0,0,0,0,0.352489,0.000000,0.000000,17.119,0.352489,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,17.395662,18.991030,Other VAHO,205905.0,NaN,NaN,NaN,NaN
85904,AURG_NHR_AD_HO,2023-04-30,17.636000,24.384833,27.020710,22.004033,0.363134,0.502096,0.556370,0.453074,GT,AURG,NHR_AD_HO,14.380,0,0,0,0,0,0,0.296091,0.000000,0.000000,14.380,0.296091,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,28.838145,30.224309,Other VAHO,205905.0,17.636000,NaN,NaN,NaN
85905,AURG_NHR_AD_HO,2023-05-31,18.491333,24.384833,36.776411,31.387400,0.380746,0.502096,0.757245,0.646282,GT,AURG,NHR_AD_HO,38.192,0,0,0,0,0,0,0.786392,0.000000,0.000000,38.192,0.786392,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,37.907949,39.415760,Other VAHO,205905.0,18.491333,NaN,NaN,NaN
85906,AURG_NHR_AD_HO,2023-06-30,23.230333,24.384833,35.831749,28.661400,0.478324,0.502096,0.737794,0.590153,GT,AURG,NHR_AD_HO,40.829,0,0,0,0,0,0,0.840690,0.000000,0.000000,40.829,0.840690,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,37.364392,38.774465,Other VAHO,205905.0,23.230333,NaN,NaN,NaN
85907,AURG_NHR_AD_HO,2023-07-31,31.133667,24.384833,17.515619,18.947467,0.641058,0.502096,0.360655,0.390138,GT,AURG,NHR_AD_HO,20.030,0,0,0,0,0,1,0.412428,0.000000,0.000000,20.030,0.412428,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,19.189461,20.569549,Other VAHO,205905.0,31.133667,24.384833,NaN,NaN
85908,AURG_NHR_AD_HO,2023-08-31,33.017000,25.754167,19.262087,19.521467,0.679837,0.530291,0.396616,0.401957,GT,AURG,NHR_AD_HO,20.448,0,0,0,0,1,0,0.421035,0.000000,0.000000,20.448,0.421035,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,20.874449,22.234344,Other VAHO,205905.0,33.017000,25.754167,NaN,NaN
85909,AURG_NHR_AD_HO,2023-09-30,27.102333,25.166333,23.576102,19.538333,0.558051,0.518187,0.485444,0.402304,GT,AURG,NHR_AD_HO,20.766,0,0,1,1,0,0,0.427582,0.000000,0.000000,20.766,0.427582,2026-02-28,0.415689,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,24.824059,26.376141,Other VAHO,205905.0,27.102333,25.166333,NaN,NaN
85910,AURG_NHR_AD_HO,2023-10-31,20.414667,25.774167,18.249999,21.088400,0.420348,0.530703,0.375777,0.434221,GT,AURG,NHR_AD_HO,22.059,0,1,0,0,0,0,0.454206,0.000000,0.000000,22.059,0.454206,

In [373]:
trend_file_df['LY P3M_copy'] = trend_file_df['LY P3M'].copy()

In [374]:
# trend_file_df['P3M'].sum()
trend_file_df[trend_file_df['month_date'] == '2026-04-30']['P3M'].sum()

3721282.006333374

In [375]:
# pd.Series([100, 2, 4, 6]).nlargest(2).mean()

In [376]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'brand_code',
       'sec_vol_actuals_rum_month', 'diwali', 'diwali_lead_1', 'diwali_lead_2',
       'ganesh_chaturthi', 'ganesh_chaturthi_lead_1',
       'ganesh_chaturthi_lead_2', 'sec_vol_actuals_rum_month_value',
       'pred_best_model', 'pred_value_best_model',
       'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov', 'run',
       'step', 'file_path', 'drive', 'outlier', 'run_month', 'M month',
       'pred_prophet_60%ile', 'pred_prophet_70%ile', 'portfolio',
       'qtr_ind_rate', 'P3M', 'P6M', 'LY P3M', 'LY P6M', 'LY P3M_copy'],
      dtype='object')

In [377]:
trend_file_df['P3M Max'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()

In [378]:
trend_file_df['P3M Top 2 Mean'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False

In [379]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'channel', 'key', 'month_date']
)

In [380]:
trend_file_df['MoM P3M growth'] = (
    trend_file_df.groupby(['run_month', 'channel', 'key'])['P3M']
      .pct_change() * 100
)

In [381]:
trend_file_df = trend_file_df.sort_values(
    ['channel', 'run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['MoM P3M growth_lag_1'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
trend_file_df['MoM P3M growth_lag_2'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)



In [382]:
trend_file_df['>=20%_3M_inc_month_count'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 

In [383]:
trend_file_df['Avg(P3M Mean, Max)'] = trend_file_df[['P3M', 'P3M Max']].mean(axis=1)

In [384]:
trend_file_df['month_date'].unique()

<DatetimeArray>
['2023-01-31 00:00:00', '2023-02-28 00:00:00', '2023-03-31 00:00:00',
 '2023-04-30 00:00:00', '2023-05-31 00:00:00', '2023-06-30 00:00:00',
 '2023-07-31 00:00:00', '2023-08-31 00:00:00', '2023-09-30 00:00:00',
 '2023-10-31 00:00:00', '2023-11-30 00:00:00', '2023-12-31 00:00:00',
 '2024-01-31 00:00:00', '2024-02-29 00:00:00', '2024-03-31 00:00:00',
 '2024-04-30 00:00:00', '2024-05-31 00:00:00', '2024-06-30 00:00:00',
 '2024-07-31 00:00:00', '2024-08-31 00:00:00', '2024-09-30 00:00:00',
 '2024-10-31 00:00:00', '2024-11-30 00:00:00', '2024-12-31 00:00:00',
 '2025-01-31 00:00:00', '2025-02-28 00:00:00', '2025-03-31 00:00:00',
 '2025-04-30 00:00:00', '2025-05-31 00:00:00', '2025-06-30 00:00:00',
 '2025-07-31 00:00:00', '2025-08-31 00:00:00', '2025-09-30 00:00:00',
 '2025-10-31 00:00:00', '2025-11-30 00:00:00', '2025-12-31 00:00:00',
 '2026-01-31 00:00:00', '2026-02-28 00:00:00', '2026-03-31 00:00:00',
 '2026-04-30 00:00:00', '2026-05-31 00:00:00', '2026-06-30 00:00:00',
 '20

In [385]:
pd.set_option('display.max_columns', None)

In [386]:
#trend_file_df['LY P3M'].sum()
trend_file_df[trend_file_df['month_date'] == '2026-02-28']['P3M'].sum()

5859927.506000042

In [387]:
trend_file_df[
    (trend_file_df['month_date'] == '2026-02-28') &
    (trend_file_df['P3M'].isna())
]

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,brand_code,sec_vol_actuals_rum_month,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,drive,outlier,run_month,M month,pred_prophet_60%ile,pred_prophet_70%ile,portfolio,qtr_ind_rate,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)"


In [388]:
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['key','channel'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [389]:
trend_file_df[trend_file_df['month_date'] == '2026-02-28']['P3M'].sum()

5859927.506000042

In [390]:
trend_file_df['LY P3M'].sum()

181566805.6896683

In [391]:
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    trend_file_df[f'{col}_value'] = trend_file_df[col] * trend_file_df['qtr_ind_rate'] / (10 ** 7)

In [392]:
trend_file_df['sec_vol_actuals_rum_month_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['sec_vol_actuals_rum_month']/ (10 ** 7)
trend_file_df['pred_prophet_70%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_70%ile']/ (10 ** 7)
trend_file_df['pred_prophet_60%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_60%ile']/ (10 ** 7)

In [393]:
value_cols = [col for col in trend_file_df.columns if 'value' in col]
value_cols

['pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'sec_vol_actuals_rum_month_value',
 'pred_value_best_model',
 'sec_vol_actuals_rum_month_value_treated',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'pred_prophet_60%ile_value']

In [394]:
for col in value_cols:
    try:
        assert trend_file_df[col].min() >= 0
    except:
        print(col)

    # trend_file_df[col] = trend_file_df[col] / (10 ** 7)

In [395]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,brand_code,sec_vol_actuals_rum_month,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,drive,outlier,run_month,M month,pred_prophet_60%ile,pred_prophet_70%ile,portfolio,qtr_ind_rate,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value
18422,BCE1_ADV-AHO-R,2023-01-31,15.00,12.05,22.887669,15.2760,0.000675,0.000542,0.001030,0.000688,ECOM,BCE1,ADV-AHO-R,18.0,0,0,0,0,0,0,0.000810,0.000,0.000000,18.0,0.000810,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,25.407847,27.861855,Hair Oils,450.074080,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001254,0.001144
18423,BCE1_ADV-AHO-R,2023-02-28,15.00,12.05,10.900028,15.1320,0.000675,0.000542,0.000491,0.000681,ECOM,BCE1,ADV-AHO-R,9.0,0,0,0,0,0,0,0.000405,0.000,0.000000,9.0,0.000405,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,13.063876,15.458126,Hair Oils,450.074080,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000696,0.000588
18424,BCE1_ADV-AHO-R,2023-03-31,15.00,12.05,7.686881,12.1800,0.000675,0.000542,0.000346,0.000548,ECOM,BCE1,ADV-AHO-R,18.0,0,0,0,0,0,0,0.000810,0.000,0.000000,18.0,0.000810,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,10.294241,12.837634,Hair Oils,450.074080,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000578,0.000463
18425,BCE1_ADV-AHO-R,2023-04-30,15.00,12.05,2.505327,10.0308,0.000675,0.000542,0.000113,0.000451,ECOM,BCE1,ADV-AHO-R,0.0,0,0,0,0,0,0,0.000000,0.000,0.000000,0.0,0.000000,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,4.683974,7.037807,Hair Oils,450.074080,15.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.00,0.000675,NaN,NaN,NaN,0.000317,0.000211
18426,BCE1_ADV-AHO-R,2023-05-31,9.00,12.05,13.166723,16.1580,0.000405,0.000542,0.000593,0.000727,ECOM,BCE1,ADV-AHO-R,12.9,0,0,0,0,0,0,0.000581,0.000,0.000000,12.9,0.000581,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,15.207517,17.602301,Hair Oils,450.074080,9.00,NaN,NaN,NaN,NaN,NaN,NaN,-40.000000,NaN,NaN,NaN,9.00,0.000405,NaN,NaN,NaN,0.000792,0.000684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18417,QCW2_SW_SGPRF,2026-11-30,11.76,5.88,0.000000,2.0800,0.001697,0.000849,0.000000,0.000300,QCOM,QCW2,SW_SGPRF,0.0,1,0,0,0,0,0,0.000000,2.080,0.000300,0.0,0.000000,2026-02-28,1.203350,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,M+8,0.000000,0.000000,Male Grooming,1443.400363,11.76,5.88,49.28,5.12,0.64,6.96,6.96,68.965517,0.0,inf,0.0,9.36,0.001697,0.000849,0.007113,0.000739,0.000000,0.000000
18418,QCW2_SW_SGPRF,2026-12-31,11.76,5.88,0.000000,10.5840,0.001697,0.000849,0.000000,0.001528,QCOM,QCW2,SW_SGPRF,0.0,0,0,0,0,0,0,0.000000,10.584,0.001528,0.0,0.000000,2026-02-28,1.203350,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,M+9,0.000000,0.000000,Male Grooming,1443.400363,11.76,5.88,49.28,3.84,0.00,6.96,6.96,68.965517,0.0,inf,0.0,9.36,0.001697,0.000849,0.007113,0.000554,0.000000,0.000000
18419,QCW2_SW_SGPRF,2027-01-31,11.76,5

In [396]:
assert trend_file_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'channel', 'month_date']
).sum() == 0

In [397]:
trend_file_df = trend_file_df.sort_values(
    ['channel', 'run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['LY'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month'].shift(12)

trend_file_df['LLY'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month'].shift(24)


trend_file_df['LY value'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(12)

trend_file_df['LLY value'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(24)


trend_file_df['Sec_Value_in_Cr_lag_1'] = trend_file_df.groupby(
    ['channel','run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(1)

trend_file_df['Sec_Value_in_Cr_lag_2'] = trend_file_df.groupby(
    ['channel','run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(2)

trend_file_df['Sec_Value_in_Cr_lag_3'] = trend_file_df.groupby(
    ['channel','run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(3)

In [398]:
trend_file_df[trend_file_df['M month']!=0][['channel','brand_code','key','month_date','LY','P3M', 'P6M', 'LY P3M', 'LY P6M'
                                            ]].to_csv('p3m_ly_corrected.csv')#[trend_file_df['brand_code'] == 'NHR_AD_HO']

In [219]:
for col in ['Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2', 'Sec_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['channel','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [220]:
# trend_file_df[['ASM', 'Depot', 'PSKU']] = trend_file_df['key'].str.split('_', expand=True)

In [221]:
trend_file_df.reset_index(drop=True, inplace=True)

In [222]:
trend_file_df.shape

(178628, 63)

In [223]:
trend_file_df['key'].nunique()

3597

In [224]:
brand_class = pd.read_excel('/data/aman_singh/acuuracy_check/brand_class_new.xlsx')
brand_class['Channel'] = brand_class['Channel'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
brand_class.columns = brand_class.columns.str.lower()
brand_class = brand_class[['channel', 'brand','final class']]
brand_class.rename(columns = {'brand':'brand_code','final class':'class'}, inplace = True)


In [225]:
# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()

In [226]:
# brand_class_df.columns = ['brand_code', 'class']

In [227]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_class, 
    on=['channel','brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)
del len_before_merge

In [228]:
# trend_file_df['class'].isna().sum()

In [229]:
trend_file_df['brand_code'].unique()

array(['ADV-AHO-R', 'BIO OILS', 'BRD_BDSPR', 'BRD_DOGAS', 'BRD_FSWSH',
       'BRD_HROIL', 'BRD_HRWAX', 'CO_SO_BL', 'CO_SO_PCP', 'CO_SO_VCN',
       'H&C', 'H&C_ALMND', 'HC SNS', 'HC_PBHOIL', 'KAYA_GM', 'KAYA_ML',
       'LIVON', 'LIVON S-R', 'LVNPST_ML', 'LVN_PRFSR', 'LVN_SRSNS',
       'LVN_SR_DR', 'MALO-NATU', 'MALT-NATU', 'NHR-SABDM', 'NHR-UTTAM',
       'NHR_AMLGD', 'NIHAR NHO', 'PA-ALO-HO', 'PA-BDYLOT', 'PABABY_CM',
       'PABABY_GM', 'PABABY_ML', 'PADV-HOT', 'PADV-HRCR', 'PADVJAS-R',
       'PADV_SMPN', 'PA_CN_HO', 'PA_EXT_ML', 'PA_HR_MSK', 'PA_JASGLD',
       'PA_ONI_HO', 'PCNO FLEX', 'PCNO(R)', 'PURSNS_ML', 'P_AL_GOLD',
       'P_EN_ALM', 'P_EN_ARG', 'P_EN_BGHB', 'P_EN_CRSH', 'P_EN_RSMR',
       'REV.LQDST', 'REV.ST.', 'REV_LQFRG', 'SAF-MUSLI', 'SAFF ACTV',
       'SAFF GOLD', 'SAFF KO', 'SAFF KOCO', 'SAFF OATS', 'SAFF SALT',
       'SAFF_ODLS', 'SAF_HONEY', 'SAF_MAYO', 'SAF_MILET', 'SAF_PNBTR',
       'SFOATS-FL', 'SFOATS_GD', 'SF_IM_CHY', 'SF_MNCHPS', 'SF_SOYACN',
       'S

In [230]:
trend_file_df[trend_file_df['class'].isna()]['brand_code'].unique()

array([], dtype=object)

In [231]:
# trend_file_df['final class'].unique()

In [232]:
trend_file_df[
    # (trend_file_df['channel'].isin(['MT', 'QCOM'])) & 
    # (trend_file_df['month_date'] > '2024-06-30') &
    (trend_file_df['M month'].notna()) 
    # (trend_file_df['class'].isin(['B', 'C']))
].shape# .to_csv('Heuristic_GT_ECOM_B&C_v1.csv', index=False)

(178628, 64)

In [233]:
(324496, 61)

(324496, 61)

In [234]:
# trend_file_df[
#     # (trend_file_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (trend_file_df['month_date'] > '2024-06-30') &
#     (trend_file_df['M month'].notna())
#     # (trend_file_df['class'].isin(['B', 'C']))
# ]['Skipped'].unique()

In [235]:
# trend_file_df[
#     # (trend_file_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (trend_file_df['month_date'] > '2024-06-30') &
#     (trend_file_df['M month'].notna())
#     # (trend_file_df['class'].isin(['B', 'C']))
# ].to_csv("ALL Channels Live Run Heuristics Dec'25.csv", index=False)

In [236]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,brand_code,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,drive,outlier,run_month,M month,pred_prophet_60%ile,pred_prophet_70%ile,portfolio,qtr_ind_rate,sec_vol_actuals_rum_month,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,class
0,BCE1_ADV-AHO-R,2023-01-31,15.00,12.05,22.887669,15.2760,0.000675,0.000542,0.001030,0.000688,ECOM,BCE1,ADV-AHO-R,0,0,0,0,0,0,0.001512,0.000,0.000000,18.0,0.000810,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,25.407847,27.861855,Hair Oils,450.074080,33.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001254,0.001144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B
1,BCE1_ADV-AHO-R,2023-02-28,15.00,12.05,10.900028,15.1320,0.000675,0.000542,0.000491,0.000681,ECOM,BCE1,ADV-AHO-R,0,0,0,0,0,0,0.001364,0.000,0.000000,9.0,0.000405,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,13.063876,15.458126,Hair Oils,450.074080,30.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000696,0.000588,NaN,NaN,NaN,NaN,0.001512,NaN,NaN,B
2,BCE1_ADV-AHO-R,2023-03-31,15.00,12.05,7.686881,12.1800,0.000675,0.000542,0.000346,0.000548,ECOM,BCE1,ADV-AHO-R,0,0,0,0,0,0,0.000810,0.000,0.000000,18.0,0.000810,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,10.294241,12.837634,Hair Oils,450.074080,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000578,0.000463,NaN,NaN,NaN,NaN,0.001364,0.001512,NaN,B
3,BCE1_ADV-AHO-R,2023-04-30,15.00,12.05,2.505327,10.0308,0.000675,0.000542,0.000113,0.000451,ECOM,BCE1,ADV-AHO-R,0,0,0,0,0,0,0.000230,0.000,0.000000,0.0,0.000000,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,4.683974,7.037807,Hair Oils,450.074080,5.1,27.30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.30,0.001229,NaN,NaN,NaN,0.000317,0.000211,NaN,NaN,NaN,NaN,0.000810,0.001364,0.001512,B
4,BCE1_ADV-AHO-R,2023-05-31,9.00,12.05,13.166723,16.1580,0.000405,0.000542,0.000593,0.000727,ECOM,BCE1,ADV-AHO-R,0,0,0,0,0,0,0.000581,0.000,0.000000,12.9,0.000581,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,15.207517,17.602301,Hair Oils,450.074080,12.9,17.80,NaN,NaN,NaN,NaN,NaN,NaN,-34.798535,NaN,NaN,NaN,17.80,0.000801,NaN,NaN,NaN,0.000792,0.000684,NaN,NaN,NaN,NaN,0.000230,0.000810,0.001364,B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178623,QCW2_SW_SGPRF,2026-11-30,11.76,5.88,0.000000,2.0800,0.001697,0.000849,0.000000,0.000300,QCOM,QCW2,SW_SGPRF,1,0,0,0,0,0,0.000000,2.080,0.000300,0.0,0.000000,2026-02-28,1.203350,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,M+8,0.000000,0.000000,Male Grooming,1443.400363,0.0,11.76,5.88,49.28,4.52,0.64,6.96,6.96,68.965517,0.0,inf,0.0,9.36,0.001697,0.000849,0.007113,0.000652,0.000000,0.000000,0.00,NaN,0.000000,NaN,0.002078,0.000000,0.003014,NPD
178624,QCW2_SW_SGPRF,2026-12-31,11.76,5.88,0.000000,10.5840,0.001697,0.000849,0.000000,0.001528,QCOM,QCW2,SW_SGPRF,0,0

In [237]:
trend_file_df.to_csv("/data/aman_singh/acuuracy_check/all_channels brand_asm mar'26.csv", index=False)

### some checks

In [7]:
trend_file_df = pd.read_csv("/data/aman_singh/acuuracy_check/all_channels brand_asm mar'26.csv")
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,class
0,BCE1_ADV-AHO-R,2023-01-31,15.00,12.05,22.887669,15.2760,0.000675,0.000542,0.001030,0.000688,...,0.001254,0.001144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B
1,BCE1_ADV-AHO-R,2023-02-28,15.00,12.05,10.900028,15.1320,0.000675,0.000542,0.000491,0.000681,...,0.000696,0.000588,NaN,NaN,NaN,NaN,0.001512,NaN,NaN,B
2,BCE1_ADV-AHO-R,2023-03-31,15.00,12.05,7.686881,12.1800,0.000675,0.000542,0.000346,0.000548,...,0.000578,0.000463,NaN,NaN,NaN,NaN,0.001364,0.001512,NaN,B
3,BCE1_ADV-AHO-R,2023-04-30,15.00,12.05,2.505327,10.0308,0.000675,0.000542,0.000113,0.000451,...,0.000317,0.000211,NaN,NaN,NaN,NaN,0.000810,0.001364,0.001512,B
4,BCE1_ADV-AHO-R,2023-05-31,9.00,12.05,13.166723,16.1580,0.000405,0.000542,0.000593,0.000727,...,0.000792,0.000684,NaN,NaN,NaN,NaN,0.000230,0.000810,0.001364,B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178623,QCW2_SW_SGPRF,2026-11-30,11.76,5.88,0.000000,2.0800,0.001697,0.000849,0.000000,0.000300,...,0.000000,0.000000,0.00,NaN,0.000000,NaN,0.002078,0.000000,0.003014,NPD
178624,QCW2_SW_SGPRF,2026-12-31,11.76,5.88,0.000000,10.5840,0.001697,0.000849,0.000000,0.001528,...,0.000000,0.000000,20.88,47.52,0.003014,0.006859,0.002078,0.000000,0.003014,NPD
178625,QCW2_SW_SGPRF,2027-01-31,11.76,5.88,0.000000,3.2960,0.001697,0.000849,0.000000,0.000476,...,0.000000,0.000000,0.00,67.20,0.000000,0.009700,0.002078,0.000000,0.003014,NPD
178626,QCW2_SW_SGPRF,2027-02-28,11.76,5.88,0.000000,6.6320,0.001697,0.000849,0.000000,0.000957,...,0.000000,0.000000,14.40,33.12,0.002078,0.004781,0.002078,0.000000,0.003014,NPD


In [8]:
query = """select * from TRN_MIL_DF_ASM_BRAND
where run_month = '2026-03-31' """

data = pd.read_sql(con=dev_conn, sql=query)
data.columns = data.columns.str.lower()
data

,channel,asm_area_code,brand_code,run_month,month_date,sec_vol_actuals_rum_month,extra_vol_co,other_co,price_off_co,seasonal_month_flag,diwali,ganesh_chaturthi,drive,outlier
0,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-04-30,18.0,0.0,0.0,0.0,0,0,0,NaN,NaN
1,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-05-31,9.0,0.0,0.0,0.0,0,0,0,NaN,NaN
2,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-06-30,72.0,0.0,0.0,0.0,0,0,0,NaN,NaN
3,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-07-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
4,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-08-31,0.0,0.0,0.0,0.0,0,0,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214622,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-11-30,0.0,0.0,0.0,0.0,0,1,0,NaN,NaN
214623,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-12-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
214624,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-01-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
214625,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-02-28,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN


In [9]:
data['key'] = (
    data['asm_area_code'].astype(str) + '_' +
    data['brand_code'].astype(str)
    
)
data = data[data['key'].isin(trend_file_df['key'].unique())]

In [111]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,brand_code,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,drive,outlier,run_month,M month,pred_prophet_60%ile,pred_prophet_70%ile,portfolio,qtr_ind_rate,sec_vol_actuals_rum_month,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,class
0,BCE1_ADV-AHO-R,2023-01-31,15.00,12.05,22.887669,15.2760,0.000675,0.000542,0.001030,0.000688,ECOM,BCE1,ADV-AHO-R,0,0,0,0,0,0,0.001512,0.000,0.000000,18.0,0.000810,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,25.407847,27.861855,Hair Oils,450.074080,33.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001254,0.001144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B
1,BCE1_ADV-AHO-R,2023-02-28,15.00,12.05,10.900028,15.1320,0.000675,0.000542,0.000491,0.000681,ECOM,BCE1,ADV-AHO-R,0,0,0,0,0,0,0.001364,0.000,0.000000,9.0,0.000405,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,13.063876,15.458126,Hair Oils,450.074080,30.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000696,0.000588,NaN,NaN,NaN,NaN,0.001512,NaN,NaN,B
2,BCE1_ADV-AHO-R,2023-03-31,15.00,12.05,7.686881,12.1800,0.000675,0.000542,0.000346,0.000548,ECOM,BCE1,ADV-AHO-R,0,0,0,0,0,0,0.000810,0.000,0.000000,18.0,0.000810,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,10.294241,12.837634,Hair Oils,450.074080,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000578,0.000463,NaN,NaN,NaN,NaN,0.001364,0.001512,NaN,B
3,BCE1_ADV-AHO-R,2023-04-30,15.00,12.05,2.505327,10.0308,0.000675,0.000542,0.000113,0.000451,ECOM,BCE1,ADV-AHO-R,0,0,0,0,0,0,0.000230,0.000,0.000000,0.0,0.000000,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,4.683974,7.037807,Hair Oils,450.074080,5.1,27.30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.30,0.001229,NaN,NaN,NaN,0.000317,0.000211,NaN,NaN,NaN,NaN,0.000810,0.001364,0.001512,B
4,BCE1_ADV-AHO-R,2023-05-31,9.00,12.05,13.166723,16.1580,0.000405,0.000542,0.000593,0.000727,ECOM,BCE1,ADV-AHO-R,0,0,0,0,0,0,0.000581,0.000,0.000000,12.9,0.000581,2026-02-28,0.930828,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,0,15.207517,17.602301,Hair Oils,450.074080,12.9,17.80,NaN,NaN,NaN,NaN,NaN,NaN,-34.798535,NaN,NaN,NaN,17.80,0.000801,NaN,NaN,NaN,0.000792,0.000684,NaN,NaN,NaN,NaN,0.000230,0.000810,0.001364,B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178623,QCW2_SW_SGPRF,2026-11-30,11.76,5.88,0.000000,2.0800,0.001697,0.000849,0.000000,0.000300,QCOM,QCW2,SW_SGPRF,1,0,0,0,0,0,0.000000,2.080,0.000300,0.0,0.000000,2026-02-28,1.203350,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,2026-03-31,M+8,0.000000,0.000000,Male Grooming,1443.400363,0.0,11.76,5.88,49.28,4.52,0.64,6.96,6.96,68.965517,0.0,inf,0.0,9.36,0.001697,0.000849,0.007113,0.000652,0.000000,0.000000,0.00,NaN,0.000000,NaN,0.002078,0.000000,0.003014,NPD
178624,QCW2_SW_SGPRF,2026-12-31,11.76,5.88,0.000000,10.5840,0.001697,0.000849,0.000000,0.001528,QCOM,QCW2,SW_SGPRF,0,0

In [11]:
import pandas as pd

as_of_date = pd.to_datetime("2026-03-31")  # month-end for Feb 2026
data['month_date'] = pd.to_datetime(data['month_date'])
filtered = data[
    (data['channel'] == 'GT') &
    (data['month_date'] < as_of_date) &
    (data['month_date'] >= as_of_date - pd.DateOffset(months=3))
]


In [12]:
# assert p3m equals
x = filtered.groupby(['month_date'])['sec_vol_actuals_rum_month'].sum().reset_index()['sec_vol_actuals_rum_month'].mean()
y = trend_file_df[(trend_file_df['month_date'] == '2026-04-30') & (trend_file_df['channel'] == 'GT')]['P3M'].sum()
assert(int(x)==int(y))

AssertionError: 

In [13]:
(x,y)

(4714701.811333333, 4706624.03499999)

In [14]:
ly_end = as_of_date - pd.DateOffset(years=1)
ly_start = ly_end - pd.DateOffset(months=3)

filtered = data[
    (data['channel'] == 'GT') &
    (data['month_date'] < ly_end) &
    (data['month_date'] >= ly_start )
]


In [15]:
# p3m ly check may not equal but should be close
x = filtered.groupby(['month_date'])['sec_vol_actuals_rum_month'].sum().reset_index()['sec_vol_actuals_rum_month'].mean()
y = trend_file_df[(trend_file_df['month_date'] == '2026-04-30') & (trend_file_df['channel'] == 'GT')]['LY P3M'].sum()
(x,y)

(3676068.8533333335, 3672362.1933333403)

In [128]:
# p3m consistency check
as_of_date = pd.to_datetime('2026-03-31')

next_3_months = pd.date_range(
    start=as_of_date + pd.offsets.MonthEnd(1),
    periods=3,
    freq='M'
)
for dt in next_3_months:
    p3m_sum = trend_file_df.loc[
        trend_file_df['month_date'] == dt, 'P3M'
    ].sum()
    
    print(f"P3M sum for {dt.date()}: {p3m_sum}")


P3M sum for 2026-04-30: 595197.0799999993
P3M sum for 2026-05-31: 595197.0799999993
P3M sum for 2026-06-30: 595197.0799999993


In [ ]:
trend_file_df[trend_file_df['month_date'] == '2026-02-28']['P3M'].sum()


624516.5986666783